# ЛР-1. Задание 2. Данные: карточки мониторов DNS (предварительный анализ)

Ноутбук запускается дважды: на всём собранном наборе (`HEAD = None`) и на пилоте, то есть первых 100 собранных карточках (`HEAD = 100`). Пилот нужен, чтобы настроить правила эталона и гипотезы, не глядя на val и test.

Анализ строится от гипотез и решений из отчёта по обзору. Каждый шаг проверяет что-то конкретное:

| Шаг | Что делаем | Гипотеза или решение |
|---|---|---|
| 1. Паспорт | размер выборки, длины, бренды | контекст для порогов |
| 2–3. Значение в тексте | для 7 атрибутов проверяем, названо ли значение из таблицы DNS в описании | решение «разметка реального теста»; метрики по каждому атрибуту |
| 4. Эталон | значение из таблицы, если оно есть в тексте, иначе `null` | H1 и метрики (доля ложных значений на `null`) |
| 5. Форматы записи | как одно и то же значение пишут в тексте | H5 (нормализация) |
| 6. Проверка карточек | расхождение с текстом (смарт-мониторы убраны в начале) | слабая разметка |
| 7. Дубликаты | почти одинаковые описания | защита от утечки между real-val и real-test |
| 8. Разбиение | real-val / real-test, seed 42 | протокол эксперимента |

Выбросов по длине или редким значениям не анализируем: гипотезы об их влиянии нет.

Чтобы запустить на другом файле, поменяй `DATA` в первой ячейке с кодом.

In [ ]:
import json, os, re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = "../data/dns_cards_500.json"
HEAD = None    # например 100: взять только первые 100 собранных карточек (пилот)
STEM = os.path.splitext(os.path.basename(DATA))[0] + (f"_first{HEAD}" if HEAD else "")
OUT = f"outputs/{STEM}"          # результаты каждого запуска лежат в своей папке
os.makedirs(OUT, exist_ok=True)

df = pd.DataFrame(json.load(open(DATA, encoding="utf-8")))
n_all = len(df)
df = df[df["specs"].apply(lambda s: s.get("Тип") == "монитор")].reset_index(drop=True)   # смарт-мониторы убираем: у них другие характеристики
print("убрано не мониторов (смарт-мониторы):", n_all - len(df))
if HEAD: df = df.head(HEAD)
df["бренд"] = df["category"].str.split(" / ").str[-1]
df["длина описания"] = df["description"].str.len()
df["число характеристик"] = df["specs"].apply(len)
print(len(df), "карточек,", df["url"].nunique(), "уникальных ссылок")

## 1. Паспорт данных

In [ ]:
display(df[["длина описания", "число характеристик"]].describe().round(0))
display(df["бренд"].value_counts())

## 2. Атрибуты и поиск значения в тексте

Для каждого атрибута задано три вещи: параметр в таблице DNS, как привести значение к единому виду
и как искать значение в тексте. Если в тексте нашлось то же значение — берём его в эталон.
Если нашлось другое — это расхождение, его проверяем руками. Если не нашлось — эталон `null`.

In [ ]:
def number(s):
    return re.search(r"\d+(?:[.,]\d+)?", s).group().replace(",", ".")

def resolution(s):
    w, h = re.findall(r"\d+", s)[:2]
    return f"{w}x{h}" if int(w) >= 640 else None   # отсекаем «100x100» (размер VESA)

def contrast(s):
    s = re.sub(r"(?<=\d)\s(?=\d{3}\b)", "", s.lower())      # «1 000 000» -> «1000000»
    v = float(number(s)) * (10**6 if re.search(r"m|м", s) else 1)   # «1.5M», «1,5 млн» -> миллионы
    return str(int(v)) if v >= 100 else None

ATTRS = {
    # атрибут: (параметр в таблице, приведение к единому виду, поиск в тексте)
    "диагональ":     ("Диагональ экрана (дюйм)",            lambda s: f"{float(number(s)):g}",
                      r'(\d+(?:[.,]\d+)?)\s*-?\s*(?:"|”|″|дюйм)'),
    "разрешение":    ("Максимальное разрешение",            resolution,
                      r"(\d{3,4}\s*[xх×*]\s*\d{3,4})"),
    "частота":       ("Максимальная частота обновления экрана", lambda s: number(s),
                      r"(\d{2,3})\s*(?:гц|герц)"),
    "тип матрицы":   ("Тип матрицы",                        lambda s: s.upper().replace("QD-", ""),
                      r"\b(ips|va|(?:qd-)?oled|tn)\b"),
    "яркость":       ("Яркость",                            lambda s: number(s),
                      r"(\d{3,4})\s*(?:кд|нит|cd)"),
    "контрастность": ("Контрастность",                      contrast,
                      r"((?:\d{1,3}(?:\s\d{3})+|\d+(?:[.,]\d+)?)\s*(?:m|млн)?)\s*(?::|к|/)\s*1\b"),
    "изогнутость":   ("Изогнутый экран",                    lambda s: s.lower(),
                      r"(изогнут|curved|плоск)"),
}
CURVED = {"изогнут": "да", "curved": "да", "плоск": "нет"}

def values_in_text(attr, text):
    """все значения атрибута, которые названы в тексте"""
    key, norm, pattern = ATTRS[attr]
    found = set()
    for m in re.findall(pattern, text):
        v = CURVED[m] if attr == "изогнутость" else norm(m)
        if v: found.add(v)
    return found

rows = []
for _, c in df.iterrows():
    text = c["description"].lower().replace("ё", "е")
    for attr, (key, norm, pattern) in ATTRS.items():
        if key not in c["specs"]:
            continue
        spec = norm(c["specs"][key])
        found = values_in_text(attr, text)
        status = "в тексте" if spec in found else "другое значение" if found else "не упомянут"
        rows.append({"url": c["url"], "title": c["title"], "атрибут": attr, "в таблице": spec,
                     "в тексте": ", ".join(sorted(found)), "статус": status,
                     "эталон": spec if status == "в тексте" else None})
pairs = pd.DataFrame(rows)
print(len(pairs), "пар «карточка–атрибут»,", pairs["эталон"].notna().sum(), "из них попали в эталон")

## 3. Как часто значение из таблицы есть в тексте

`±` — полуширина 95% доверительного интервала (Уилсон) для доли «в тексте». На малой выборке она большая.

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / d
    return centre - half, centre + half

cov = pairs.pivot_table(index="атрибут", columns="статус", values="url", aggfunc="count", fill_value=0)
for s in ["в тексте", "другое значение", "не упомянут"]:
    if s not in cov: cov[s] = 0
cov = cov[["в тексте", "другое значение", "не упомянут"]]
cov["всего"] = cov.sum(axis=1)
cov["доля в тексте"] = (cov["в тексте"] / cov["всего"]).round(2)
cov["±"] = [round((lambda lo_hi: (lo_hi[1] - lo_hi[0]) / 2)(wilson(r["в тексте"], r["всего"])), 2) for _, r in cov.iterrows()]
display(cov.sort_values("доля в тексте", ascending=False))

(cov[["в тексте", "другое значение", "не упомянут"]].sort_values("в тексте")
    .plot.barh(stacked=True, figsize=(8, 3.5), color=["#4c9a6a", "#d9a441", "#b8bec7"]))
plt.xlabel("число карточек"); plt.ylabel("")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)   # легенда под графиком
plt.tight_layout()
plt.savefig(f"{OUT}/coverage.png", dpi=150)
plt.show()

### Расхождения между текстом и таблицей
Их нужно проверить вручную: либо в таблице ошибка, либо в тексте названо другое (например, другая частота).

In [ ]:
display(pairs[pairs["статус"] == "другое значение"][["title", "атрибут", "в таблице", "в тексте"]])

да, действительно ошибка

## 4. Эталон
В `03_eda/outputs/<имя файла>/gold_pairs.csv` лежат только пары без текстов (его можно коммитить).
В `data/<имя файла>_gold.json` — карточки с описанием и эталоном (в `.gitignore`).

In [ ]:
pairs.drop(columns=["title"]).to_csv(f"{OUT}/gold_pairs.csv", index=False, encoding="utf-8")

gold = {u: {r["атрибут"]: (r["эталон"] if pd.notna(r["эталон"]) else None) for _, r in g.iterrows()}
        for u, g in pairs.groupby("url")}
cards_gold = [{"url": c["url"], "title": c["title"], "description": c["description"], "gold": gold.get(c["url"], {})}
              for _, c in df.iterrows()]
json.dump(cards_gold, open(f"../data/{STEM}_gold.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("пример эталона:", cards_gold[0]["title"], cards_gold[0]["gold"])

## 5. Форматы записи значений
В таблице DNS значения записаны единообразно, а в описаниях по-разному. Это проверка гипотезы H5: ошибки формата нужно считать отдельно от ошибок извлечения.

In [ ]:
corpus = " ".join(df["description"])
low = corpus.lower()
formats = {
    "диагональ: 27\"":            len(re.findall(r'\d"', corpus)),
    "диагональ: 27-дюймовый":     len(re.findall(r"\d[.,]?\d*\s*-\s*дюйм", low)),
    "диагональ: 27 дюймов":       len(re.findall(r"\d\s*дюйм", low)),
    "дробная часть через точку":   len(re.findall(r'\d\.\d+\s*-?\s*(?:"|дюйм)', low)),
    "дробная часть через запятую": len(re.findall(r'\d,\d+\s*-?\s*(?:"|дюйм)', low)),
    "разрешение: 1920x1080 (латинская x)":  len(re.findall(r"\d{3,4}\s*x\s*\d{3,4}", low)),
    "разрешение: 1920х1080 (кириллическая х)": len(re.findall(r"\d{3,4}\s*х\s*\d{3,4}", low)),
}
display(pd.Series(formats, name="упоминаний в описаниях").to_frame())

## 6. Проверка карточек
Карточки не удаляются и по выбросам не анализируются. Смарт-мониторы убраны в начале (у них другие характеристики). Здесь смотрим только расхождение с текстом: в тексте названо не то значение, что в таблице. Такие карточки проверяем руками (слабая разметка).

In [ ]:
conflict_urls = set(pairs.loc[pairs["статус"] == "другое значение", "url"])
check = pd.DataFrame({"title": df["title"]})
check["расхождение с текстом"] = df["url"].isin(conflict_urls)
check["проверить"] = check["расхождение с текстом"]
print("карточек на ручную проверку:", check["проверить"].sum(), "из", len(check))
display(check[check["проверить"]])

## 7. Почти-дубликаты
Косинусная близость по символьным n-граммам TF-IDF. Порог 0,97 из протокола.

In [ ]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform(df["description"])
sim = cosine_similarity(tfidf)
pairs_sim = sorted(((sim[i, j], i, j) for i in range(len(df)) for j in range(i + 1, len(df))), reverse=True)
print("самая похожая пара:", round(pairs_sim[0][0], 3), df["title"][pairs_sim[0][1]], "|", df["title"][pairs_sim[0][2]])
dups = [(i, j) for s, i, j in pairs_sim if s >= 0.97]
print("пар с близостью >= 0.97:", len(dups))

## 8. Разбиение на pilot, real-val и real-test
Из каждой пары дубликатов оставляем одну карточку. Первые `N_PILOT` собранных карточек идут на пилот (порядок сбора случайный), остальные делятся случайно (seed 42) на real-val и real-test. Тест открывается один раз в конце.
Если запустить с `HEAD = 100` (анализ пилота), на val и test ничего не останется: это нормально.

In [ ]:
VAL_SHARE, SEED, N_PILOT = 0.4, 42, 100
kept = df.drop(index=list({j for i, j in dups}))        # дубликаты убираем до разбиения
pilot = kept.iloc[:N_PILOT]                              # первые собранные карточки (порядок сбора случайный) идут на пилот
rest = kept.iloc[N_PILOT:].sample(frac=1, random_state=SEED)
n_val = round(len(rest) * VAL_SHARE)
split = {"pilot": pilot["url"].tolist(), "real_val": rest["url"].iloc[:n_val].tolist(), "real_test": rest["url"].iloc[n_val:].tolist()}
json.dump(split, open(f"{OUT}/split_ids.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print({k: len(v) for k, v in split.items()}, "| убрано дубликатов:", len(df) - len(kept))